# Workshop 4: Retrieve Context
Executed with free local all-MiniLM-L6-v2 embeddings (384 dimensions). No Gemini or LLM calls.
Run scripts/validate_retrieval.py from the project root first to create the isolated .chroma-local database.
Retriever is a thin boundary for future reranking, filtering, and query rewriting.


In [1]:
import os
import sys
from pathlib import Path

root = Path.cwd() if Path("src/rag").exists() else Path.cwd().parent
sys.path.insert(0, str(root))
os.chdir(root)

In [2]:
from scripts.validate_retrieval import LocalValidationEmbeddings

from rag.retrieval.retriever import Retriever
from rag.vectorstore.chroma_store import ChromaStore

store = ChromaStore(LocalValidationEmbeddings(), persist_dir=Path(".chroma-local"))
assert store.count() == 47
retriever = Retriever(store)
print(f"Persistent local vectors: {store.count()}")

Persistent local vectors: 47


In [3]:
for question in [
    "What is retrieval augmented generation?",
    "How does ChromaDB persist vectors?",
    "What is the Apache License?",
]:
    print(question)
    for chunk in retriever.search(question, k=3):
        print(Path(chunk.source).name, chunk.chunk_index, chunk.text[:100].replace("\n", " "))

What is retrieval augmented generation?
rag_primer.md 0 # Retrieval-Augmented Generation: a one-page primer  Retrieval-Augmented Generation (RAG) is a techn
rag_primer.md 3 o look next  - Lewis et al. 2020 — the original "Retrieval-Augmented Generation" paper. - The `chrom
rag_paper_summary.pdf 0 Retrieval-Augmented Generation for  Knowledge-Intensive NLP Tasks A reading summary of Lewis et al. 
How does ChromaDB persist vectors?
chromadb_overview.txt 0 ChromaDB at a glance ====================  ChromaDB is an open-source vector database that runs as a
rag_primer.md 3 o look next  - Lewis et al. 2020 — the original "Retrieval-Augmented Generation" paper. - The `chrom
embeddings_explained.md 3 for modern transformer-based embeddings | | L2 (Euclidean) | Straight-line distance | OK if your vec
What is the Apache License?
apache_foundation_faq.md 7 the foundation's website - <https://incubator.apache.org> — the Apache Incubator - <https://www.apac
apache_foundation_faq.md 2 e Incubator be

In [4]:
source = "data/workshop_corpus/apache_foundation_faq.md"
# Source separators are platform-dependent: use the stored value.
source = next(
    c.source
    for c in retriever.search("Apache License", k=5)
    if Path(c.source).name == "apache_foundation_faq.md"
)
results = retriever.search("license", k=3, filters={"source": source})
assert results and all(c.source == source for c in results)
print(f"Source-filtered results: {len(results)}; all from {source}")

Source-filtered results: 3; all from data\workshop_corpus\apache_foundation_faq.md
